# Final Round: Reinforcement Learning & Pure Exploitation

## Strategy: The Greedy Agent (Pure Exploitation)
In Reinforcement Learning, as the episode nears its end, an agent must shift from exploration (gathering info) to pure exploitation (maximizing immediate reward). Since this is our final query, we will:
1. **Zero Exploration:** Set the UCB parameter $\kappa = 0.0$. We no longer care about uncertainty.
2. **Greedy Q-Value Maximization:** We rely entirely on the predicted mean of our surrogate model (our Q-function).
3. **Micro Trust Region:** We shrink the boundaries to a tiny radius (e.g., 5%) around the absolute best point we've ever found to perform final local fine-tuning.

In [1]:
import numpy as np
import warnings
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import sys
import os

# Ensure we can import from src directory
sys.path.append(os.path.abspath('..'))
from src.utils import load_data

warnings.filterwarnings("ignore")
np.random.seed(99) # Final deterministic seed

print("Ready for Pure Exploitation (Greedy Strategy)")

Ready for Pure Exploitation (Greedy Strategy)


In [2]:
def suggest_final_point_greedy(func_id, X_train, y_train):
    print(f"\n--- Optimizing Function {func_id} (Pure Exploitation) ---")
    
    # 1. Preprocessing
    scaler_x = StandardScaler()
    X_scaled = scaler_x.fit_transform(X_train)
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
    
    # 2. Train Surrogate Models (Our "Q-Value Approximators")
    hidden_layers = (128, 64) if func_id in [7, 8] else (64, 32)
    base_mlp = MLPRegressor(hidden_layer_sizes=hidden_layers, alpha=0.01, 
                            activation='tanh', solver='lbfgs', max_iter=2000)
    
    regr = BaggingRegressor(estimator=base_mlp, n_estimators=20, random_state=42, n_jobs=-1)
    regr.fit(X_scaled, y_scaled)
    
    kernel = ConstantKernel(1.0) * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=0.1)
    gp_model = GaussianProcessRegressor(kernel=kernel, normalize_y=False)
    gp_model.fit(X_scaled, y_scaled)
    
    # 3. Micro Trust Region around the absolute best point
    best_idx = np.argmax(y_train)
    best_X_real = X_train[best_idx]
    
    # Radius is very tight: only 5% movement allowed from the best known point
    radius = 0.05
    bounds_scaled = []
    for i in range(X_train.shape[1]):
        mean_val, scale_val = scaler_x.mean_[i], scaler_x.scale_[i]
        lower = (max(0.0, best_X_real[i] - radius) - mean_val) / scale_val
        upper = (min(1.0, best_X_real[i] + radius) - mean_val) / scale_val
        bounds_scaled.append((lower, upper))
        
    best_X_scaled = scaler_x.transform(best_X_real.reshape(1, -1)).flatten()
    
    # 4. Objective Function (Pure Mean, No Uncertainty)
    def objective_function(x):
        x_reshaped = x.reshape(1, -1)
        nn_preds = np.array([est.predict(x_reshaped)[0] for est in regr.estimators_])
        avg_nn = np.mean(nn_preds)
        
        gp_pred = gp_model.predict(x_reshaped)[0]
        
        # The "Expected Reward" (Q-Value)
        comb_mean = 0.6 * avg_nn + 0.4 * gp_pred
        
        # Notice: No kappa * std. We do not care about exploration anymore.
        # Notice: Minimal repulsion, we just want to climb the immediate hill.
        dist_sq = np.sum((x_reshaped - best_X_scaled.reshape(1, -1))**2)
        penalty = 2.0 * np.exp(-dist_sq / (2 * 0.02**2)) 
        
        return -comb_mean + penalty

    # 5. Greedy Optimization step
    # Start slightly off the peak to find the true local gradient
    x_init = best_X_scaled + np.random.uniform(-0.01, 0.01, size=best_X_scaled.shape)
    res = minimize(fun=objective_function, x0=x_init, method='L-BFGS-B', 
                   bounds=bounds_scaled, options={'maxiter': 200})
    
    next_point = np.clip(scaler_x.inverse_transform(res.x.reshape(1, -1)).flatten(), 0.0, 1.0)
    return next_point

In [3]:
submission_queries = {}
print("Starting Final Round Evaluation (Greedy RL Logic)...")

for func_id in range(1, 9):
    # Update data to your current max points before running
    X_known, y_known = load_data(func_id)
    next_x = suggest_final_point_greedy(func_id, X_known, y_known)
    submission_queries[func_id] = next_x

print("\n" + "="*30)
print("FORMATTED SUBMISSION OUTPUT")
print("="*30)

for func_id, x_val in submission_queries.items():
    formatted_str = "-".join([f"{val:.6f}" for val in x_val])
    print(f"function_number: {func_id}: {formatted_str}")

Starting Final Round Evaluation (Greedy RL Logic)...

--- Optimizing Function 1 (Pure Exploitation) ---

--- Optimizing Function 2 (Pure Exploitation) ---

--- Optimizing Function 3 (Pure Exploitation) ---

--- Optimizing Function 4 (Pure Exploitation) ---

--- Optimizing Function 5 (Pure Exploitation) ---

--- Optimizing Function 6 (Pure Exploitation) ---

--- Optimizing Function 7 (Pure Exploitation) ---

--- Optimizing Function 8 (Pure Exploitation) ---

FORMATTED SUBMISSION OUTPUT
function_number: 1: 0.473240-0.470005
function_number: 2: 0.721089-0.912492
function_number: 3: 0.434270-0.524352-0.463941
function_number: 4: 0.391791-0.385698-0.403948-0.405324
function_number: 5: 1.000000-0.979736-1.000000-1.000000
function_number: 6: 0.336005-0.400861-0.565181-0.767504-0.100077
function_number: 7: 0.000000-0.280338-0.371372-0.277773-0.318762-0.675287
function_number: 8: 0.000000-0.082897-0.066149-0.148036-0.796159-0.538134-0.106415-0.679791
